# BirdCLEF 2026 — Inference v24 (PerchGRU + Mel Ensemble)
## ONNX Perch + BiGRU + ResNet18(v20) + EfficientNet-B0(v20) + Gaussian smoothing

### Architecture:
- **Perch branch (5 folds)**: ONNX `perch_v2_cpu.onnx` → PerchGRU BiGRU classifier (sequence)
- **Mel branch — ResNet18 (5 folds)**: Log-mel spectrogram at 16kHz → BirdCLEFModel(resnet18)
- **Mel branch — EfficientNet-B0 (5 folds)**: Log-mel spectrogram at 16kHz → BirdCLEFModel(efficientnet_b0)
- **Ensemble**: mean of all 15 fold predictions (5+5+5)
- **Post-processing**: Gaussian smoothing (σ=1.0) over the time axis

### Required Kaggle inputs:
1. `birdclef-2026` (competition data)
2. `chiragggg/birdclef-2026-perch-onnx` OR `vyankteshdwivedi/perch-onnx-for-birdclef+2026`
3. `chiragggg/birdclef-2026-perch-weights-v23` (PerchGRU checkpoints)
4. `chiragggg/birdclef-2026-v20-model` (ResNet18 + EfficientNet-B0 mel checkpoints)

In [ ]:
# === CELL 0: IMPORTS & CONFIG ===
import os, warnings, traceback, gc
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import onnxruntime as ort
from scipy.ndimage import gaussian_filter1d

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast

import timm
from tqdm import tqdm

warnings.filterwarnings('ignore')

CFG = dict(
    folds         = 5,
    device        = 'cuda' if torch.cuda.is_available() else 'cpu',

    # Perch / GRU branch
    perch_sr      = 32000,
    perch_seconds = 5,
    perch_emb_dim = 1536,
    perch_batch   = 16,
    gru_hidden    = 512,
    gru_layers    = 2,

    # Mel models branch (v20)
    mel_sr        = 16000,
    mel_seconds   = 5,
    n_mels        = 64,
    n_fft         = 1024,
    hop_length    = 320,
    fmin          = 60,
    fmax          = 8000,

    # Post-processing
    gauss_sigma   = 1.0,
)
CFG['perch_target'] = CFG['perch_sr'] * CFG['perch_seconds']   # 160,000
CFG['mel_target']   = CFG['mel_sr']  * CFG['mel_seconds']      # 80,000

device = torch.device(CFG['device'])
torch.set_num_threads(os.cpu_count() or 4)
print(f'Device       : {device}')
print(f'onnxruntime  : {ort.__version__}')

In [ ]:
# === CELL 1: PATHS & SPECIES ===
def _first_existing(*candidates):
    return next((p for p in candidates if os.path.exists(p)), candidates[0])

TAXONOMY_CSV   = _first_existing(
    '/kaggle/input/birdclef-2026/taxonomy.csv',
    '/kaggle/input/competitions/birdclef-2026/taxonomy.csv',
)
TEST_AUDIO     = _first_existing(
    '/kaggle/input/birdclef-2026/test_soundscapes',
    '/kaggle/input/competitions/birdclef-2026/test_soundscapes',
)
SAMPLE_SUB     = _first_existing(
    '/kaggle/input/birdclef-2026/sample_submission.csv',
    '/kaggle/input/competitions/birdclef-2026/sample_submission.csv',
)
PERCH_CKPT_DIR = _first_existing(
    '/kaggle/input/birdclef-2026-perch-weights-v23',
    '/kaggle/working',
)
MEL_CKPT_DIR   = _first_existing(
    '/kaggle/input/birdclef-2026-v20-model',
    '/kaggle/working',
)

# ONNX model
_onnx_candidates = [
    '/kaggle/input/birdclef-2026-perch-onnx/perch_v2_cpu.onnx',
    '/kaggle/input/datasets/chiragggg/birdclef-2026-perch-onnx/perch_v2_cpu.onnx',
    '/kaggle/input/perch-onnx-for-birdclef2026/perch_v2_cpu.onnx',
    '/kaggle/input/perch-onnx-for-birdclef2026/model.onnx',
]
ONNX_PATH = None
for _c in _onnx_candidates:
    if os.path.exists(_c):
        ONNX_PATH = _c
        print(f'ONNX: {ONNX_PATH}')
        break
if ONNX_PATH is None:
    import glob
    print(f'ONNX not found. Available: {glob.glob("/kaggle/input/**/*.onnx", recursive=True)}')

taxonomy_df = pd.read_csv(TAXONOMY_CSV)
species     = taxonomy_df['primary_label'].astype(str).tolist()
sp_idx      = {lab: i for i, lab in enumerate(species)}
n_classes   = len(species)

print(f'Species      : {n_classes}')
print(f'PERCH_CKPT   : {PERCH_CKPT_DIR}')
print(f'MEL_CKPT     : {MEL_CKPT_DIR}')

In [ ]:
# === CELL 2: MODEL DEFINITIONS ===

# ---- PerchGRU (Perch branch) ----
class PerchGRU(nn.Module):
    def __init__(self, n_classes: int, emb_dim: int = 1536,
                 hidden: int = 512, n_layers: int = 2, dropout: float = 0.3):
        super().__init__()
        self.proj = nn.Sequential(
            nn.LayerNorm(emb_dim),
            nn.Linear(emb_dim, 512),
            nn.GELU(),
        )
        self.gru = nn.GRU(
            input_size=512, hidden_size=hidden,
            num_layers=n_layers, batch_first=True,
            bidirectional=True,
            dropout=dropout if n_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(
            nn.LayerNorm(hidden * 2),
            nn.Dropout(0.2),
            nn.Linear(hidden * 2, n_classes),
        )
    def forward(self, x):
        z = self.proj(x)
        h, _ = self.gru(z)
        return self.head(h)  # (B, T, n_classes)


# ---- BirdCLEFModel (Mel branch — same arch as v20) ----
class BirdCLEFModel(nn.Module):
    """ResNet18 or EfficientNet-B0 mel classifier (v20 architecture)."""
    def __init__(self, arch: str, n_classes: int, pretrained: bool = False):
        super().__init__()
        self.arch = arch
        if arch == 'resnet18':
            base     = timm.create_model('resnet18', pretrained=pretrained, in_chans=1)
            n_feats  = base.fc.in_features
            base.fc  = nn.Identity()
        elif arch == 'efficientnet_b0':
            base               = timm.create_model('efficientnet_b0', pretrained=pretrained, in_chans=1)
            n_feats            = base.classifier.in_features
            base.classifier    = nn.Identity()
        else:
            raise ValueError(f'Unknown arch: {arch}')
        self.backbone = base
        self.pool   = nn.AdaptiveAvgPool2d(1)
        self.head   = nn.Sequential(
            nn.Linear(n_feats, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, n_classes),
        )
    def forward(self, x):  # x: (B, 1, n_mels, T)
        feats = self.backbone(x)
        if feats.dim() == 4:
            feats = self.pool(feats).flatten(1)
        return self.head(feats)


print('✅ PerchGRU + BirdCLEFModel defined')

In [ ]:
# === CELL 3: LOAD CHECKPOINTS (GRU + MEL MODELS) ===
def _load_ckpts(ModelClass, ckpt_names, ckpt_dir, **model_kwargs):
    models, missing = [], []
    for name in ckpt_names:
        ckpt = Path(ckpt_dir) / name
        if not ckpt.exists():
            missing.append(str(ckpt))
            continue
        m = ModelClass(**model_kwargs).to(device)
        m.load_state_dict(torch.load(ckpt, map_location=device, weights_only=True))
        m.eval()
        models.append(m)
        print(f'   ✅ {name}')
    if missing:
        for p in missing:
            print(f'   ⚠️  MISSING: {p}')
    return models


# --- Perch GRU folds ---
print('Loading PerchGRU (v23)...')
gru_models = _load_ckpts(
    PerchGRU,
    [f'perch_gru_v23_fold{i}.pt' for i in range(CFG['folds'])],
    PERCH_CKPT_DIR,
    n_classes=n_classes, emb_dim=CFG['perch_emb_dim'],
    hidden=CFG['gru_hidden'], n_layers=CFG['gru_layers'],
)

# --- ResNet18 mel folds (v20) ---
print('\nLoading ResNet18 (v20)...')
resnet_models = _load_ckpts(
    BirdCLEFModel,
    [f'resnet18_v20_fold{i}.pt' for i in range(CFG['folds'])],
    MEL_CKPT_DIR,
    arch='resnet18', n_classes=n_classes,
)

# --- EfficientNet-B0 mel folds (v20) ---
print('\nLoading EfficientNet-B0 (v20)...')
effnet_models = _load_ckpts(
    BirdCLEFModel,
    [f'efficientnet_b0_v20_fold{i}.pt' for i in range(CFG['folds'])],
    MEL_CKPT_DIR,
    arch='efficientnet_b0', n_classes=n_classes,
)

total_mel = len(resnet_models) + len(effnet_models)
print(f'\n--- Summary ---')
print(f'PerchGRU    : {len(gru_models)} folds')
print(f'ResNet18    : {len(resnet_models)} folds')
print(f'EfficientB0 : {len(effnet_models)} folds')
print(f'Total models: {len(gru_models) + total_mel}')

all_models_loaded = (len(gru_models) > 0) and (total_mel > 0)
if not all_models_loaded:
    print('\n⚠️  Some models missing — ensemble will be partial.')

In [ ]:
# === CELL 4: LOAD ONNX SESSION + SANITY CHECK ===
_ort_session  = None
_onnx_inp     = None
_onnx_emb_key = None
_onnx_ready   = False

if ONNX_PATH is None:
    print('ERROR: ONNX file not found. Perch branch will be disabled.')
else:
    try:
        _sess_opts = ort.SessionOptions()
        _sess_opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        _sess_opts.intra_op_num_threads = os.cpu_count() or 4
        _ort_session = ort.InferenceSession(
            ONNX_PATH, sess_options=_sess_opts,
            providers=['CPUExecutionProvider'],
        )
        _onnx_inp = _ort_session.get_inputs()[0].name

        # Detect 1536-d output
        for out in _ort_session.get_outputs():
            if out.shape and out.shape[-1] == 1536:
                _onnx_emb_key = out.name
                break
        if _onnx_emb_key is None:
            _test = np.zeros((1, CFG['perch_target']), dtype=np.float32)
            _outs = _ort_session.run(None, {_onnx_inp: _test})
            _names = [o.name for o in _ort_session.get_outputs()]
            for _nm, _ov in zip(_names, _outs):
                if _ov.ndim >= 2 and _ov.shape[-1] == 1536:
                    _onnx_emb_key = _nm; break
        assert _onnx_emb_key, 'Cannot find 1536-d output in ONNX model.'

        # model_info.json override
        _info_json = Path(ONNX_PATH).parent / 'model_info.json'
        if _info_json.exists():
            import json
            _info = json.load(open(_info_json))
            _onnx_inp     = _info.get('input_name', _onnx_inp)
            _onnx_emb_key = _info.get('embedding_output_name', _onnx_emb_key)

        # Sanity test
        _t = np.sin(2*np.pi*440*np.linspace(0,5,CFG['perch_target'])).astype(np.float32)[None]
        _o = _ort_session.run(None, {_onnx_inp: _t})
        _k = [x.name for x in _ort_session.get_outputs()].index(_onnx_emb_key)
        _e = _o[_k]
        if _e.ndim == 3: _e = _e.mean(1)
        assert _e[0].std() > 0.1, f'ONNX embedding std too low: {_e[0].std():.4f}'
        _onnx_ready = True
        print(f'✅ ONNX ready  input={_onnx_inp!r}  emb={_onnx_emb_key!r}')
        print(f'   Sanity emb: mean={_e[0].mean():.4f}  std={_e[0].std():.4f}')
        del _t, _o, _e
    except Exception as _ex:
        traceback.print_exc()
        print(f'\nONNX load error: {_ex}')

_out_names = [o.name for o in _ort_session.get_outputs()] if _ort_session else []
_emb_idx   = _out_names.index(_onnx_emb_key) if _onnx_emb_key in _out_names else 0
print(f'_onnx_ready  : {_onnx_ready}')

In [ ]:
# === CELL 5: MEL SPECTROGRAM HELPER ===
_mel_filter = librosa.filters.mel(
    sr=CFG['mel_sr'], n_fft=CFG['n_fft'],
    n_mels=CFG['n_mels'], fmin=CFG['fmin'], fmax=CFG['fmax'],
)


def logmel_from_wave(wave_16k: np.ndarray) -> np.ndarray:
    """
    wave_16k: (N,) float32 at 16kHz  ->  log-mel  (n_mels, T_frames)  float32
    """
    if len(wave_16k) < CFG['mel_target']:
        wave_16k = np.pad(wave_16k, (0, CFG['mel_target'] - len(wave_16k)))
    stft = librosa.stft(
        wave_16k, n_fft=CFG['n_fft'], hop_length=CFG['hop_length'],
        window='hann', center=True,
    )
    power  = np.abs(stft) ** 2
    mel    = _mel_filter @ power
    logmel = np.log1p(mel).astype(np.float32)
    return logmel


def mel_tensor_from_wave(wave_16k: np.ndarray) -> torch.Tensor:
    """Returns (1, 1, n_mels, T_frames) on device."""
    lm = logmel_from_wave(wave_16k)
    lm = (lm - lm.mean()) / (lm.std() + 1e-6)
    return torch.from_numpy(lm).float().unsqueeze(0).unsqueeze(0).to(device)


# Quick sanity check
_sine_16k = np.sin(2*np.pi*440*np.linspace(0,5,80000)).astype(np.float32)
_lm       = logmel_from_wave(_sine_16k)
print(f'✅ logmel shape: {_lm.shape}  (expected n_mels={CFG["n_mels"]} x T_frames)')
del _sine_16k, _lm

In [ ]:
# === CELL 6: PREDICTION FUNCTION ===
_use_amp = (device.type == 'cuda')


def _perch_embs(audio_path: str, end_secs_list: list) -> dict:
    """Compute Perch ONNX embeddings. Returns {end_s: np.ndarray(1536,)}."""
    result = {}
    if _ort_session is None or not end_secs_list:
        return result
    try:
        y, sr0 = sf.read(audio_path, always_2d=False)
        if y.ndim == 2: y = y.mean(1)
        if sr0 != CFG['perch_sr']:
            y = librosa.resample(y.astype(np.float32), orig_sr=sr0, target_sr=CFG['perch_sr'])
        y = y.astype(np.float32)
    except Exception as _e:
        print(f'   [WARN] perch audio load failed: {_e}')
        return result

    clips = []
    for es in end_secs_list:
        end_s  = int(es * CFG['perch_sr'])
        start  = max(0, end_s - CFG['perch_target'])
        clip   = y[start:end_s]
        if len(clip) < CFG['perch_target']:
            clip = np.pad(clip, (0, CFG['perch_target'] - len(clip)))
        clips.append(clip)

    all_embs = []
    for bi in range(0, len(clips), CFG['perch_batch']):
        batch = np.stack(clips[bi:bi+CFG['perch_batch']])
        outs  = _ort_session.run(None, {_onnx_inp: batch})
        embs  = outs[_emb_idx]
        if embs.ndim == 3: embs = embs.mean(1)
        all_embs.append(embs.astype(np.float32))

    all_embs_np = np.vstack(all_embs)
    if all_embs_np.std() < 0.05:
        print(f'   [WARN] near-zero Perch embeddings for {Path(audio_path).stem}')
    for es, emb in zip(end_secs_list, all_embs_np):
        result[es] = emb
    return result


def predict_soundscape_v24(audio_path: str, end_seconds: list) -> np.ndarray:
    """
    Ensemble prediction for one soundscape:
      - Perch+GRU (5 folds, sequence)  -> (T, n_classes)
      - ResNet18 (5 folds, per window) -> (T, n_classes)
      - EfficientNet-B0 (5 folds, per window) -> (T, n_classes)
      - Mean across all available model predictions, then Gaussian smoothing.
    Returns (T, n_classes) probabilities.
    """
    T       = len(end_seconds)
    neutral = np.full((T, n_classes), 0.5, dtype=np.float32)

    all_branch_probs = []  # list of (T, n_classes)

    # ---- Perch + GRU branch ----
    if gru_models and _onnx_ready:
        embs_map = _perch_embs(audio_path, end_seconds)
        emb_list = [
            embs_map.get(es, np.zeros(CFG['perch_emb_dim'], dtype=np.float32))
            for es in end_seconds
        ]
        emb_seq = torch.from_numpy(np.stack(emb_list)).float().unsqueeze(0).to(device)  # (1, T, 1536)
        for m in gru_models:
            with torch.inference_mode(), autocast(enabled=_use_amp):
                probs = torch.sigmoid(m(emb_seq).float())[0].cpu().numpy()  # (T, n_classes)
            all_branch_probs.append(probs)
    else:
        print('   [WARN] Perch+GRU disabled')

    # ---- Mel models branch (ResNet18 + EfficientNet-B0) ----
    mel_models = resnet_models + effnet_models
    if mel_models:
        # Load audio once at 16kHz for all mel models
        try:
            y16, sr0 = sf.read(audio_path, always_2d=False)
            if y16.ndim == 2: y16 = y16.mean(1)
            if sr0 != CFG['mel_sr']:
                y16 = librosa.resample(y16.astype(np.float32), orig_sr=sr0, target_sr=CFG['mel_sr'])
            y16 = y16.astype(np.float32)
        except Exception as _e:
            print(f'   [WARN] mel audio load failed: {_e}')
            y16 = None

        if y16 is not None:
            # Precompute mel tensors for all windows
            mel_tens_list = []
            for es in end_seconds:
                end_samp  = int(es * CFG['mel_sr'])
                start_samp = max(0, end_samp - CFG['mel_target'])
                clip       = y16[start_samp:end_samp]
                mel_tens_list.append(mel_tensor_from_wave(clip))  # (1,1,n_mels,Tf)

            mel_batch = torch.cat(mel_tens_list, dim=0)  # (T, 1, n_mels, Tf)

            for m in mel_models:
                with torch.inference_mode(), autocast(enabled=_use_amp):
                    logits = m(mel_batch).float()         # (T, n_classes)
                    probs  = torch.sigmoid(logits).cpu().numpy()  # (T, n_classes)
                all_branch_probs.append(probs)

    if not all_branch_probs:
        return neutral

    # ---- Mean ensemble ----
    probs_mean = np.mean(all_branch_probs, axis=0).astype(np.float32)  # (T, n_classes)

    # ---- Gaussian smoothing ----
    if T > 1 and CFG['gauss_sigma'] > 0:
        probs_mean = gaussian_filter1d(
            probs_mean.astype(np.float64), sigma=CFG['gauss_sigma'], axis=0
        ).astype(np.float32)

    return probs_mean


print('✅ predict_soundscape_v24() defined')
print(f'   GRU models   : {len(gru_models)}')
print(f'   ResNet18     : {len(resnet_models)}')
print(f'   EfficientB0  : {len(effnet_models)}')
print(f'   Gauss sigma  : {CFG["gauss_sigma"]}')

In [ ]:
# === CELL 7: GENERATE PREDICTIONS ===
sample_sub = pd.read_csv(SAMPLE_SUB).copy()
sample_sub['_sc_id'] = sample_sub['row_id'].str.rsplit('_', n=1).str[0]
print(f'Submission rows: {len(sample_sub)}')

all_row_ids    = []
all_probs_list = []
missing_audio  = 0
error_count    = 0

for sc_id, grp in tqdm(sample_sub.groupby('_sc_id'), desc='Soundscapes', unit='file'):
    row_ids = [str(r) for r in grp['row_id']]

    audio_path = None
    for ext in ['.ogg', '.wav', '.flac']:
        c = Path(TEST_AUDIO) / f'{sc_id}{ext}'
        if c.exists():
            audio_path = str(c); break

    if audio_path is None:
        missing_audio += 1
        all_row_ids.extend(row_ids)
        all_probs_list.append(np.full((len(row_ids), n_classes), 0.5, dtype=np.float32))
        continue

    try:
        end_seconds = [int(rid.rsplit('_', 1)[-1]) for rid in row_ids]
    except Exception as _e:
        print(f'WARNING: parse failed for {sc_id}: {_e}')
        error_count += 1
        all_row_ids.extend(row_ids)
        all_probs_list.append(np.full((len(row_ids), n_classes), 0.5, dtype=np.float32))
        continue

    try:
        probs = predict_soundscape_v24(audio_path, end_seconds)
    except Exception as _e:
        print(f'WARNING: prediction failed for {sc_id}: {_e}')
        traceback.print_exc()
        error_count += 1
        probs = np.full((len(row_ids), n_classes), 0.5, dtype=np.float32)

    all_row_ids.extend(row_ids)
    all_probs_list.append(probs)

if missing_audio:
    print(f'\n⚠️  {missing_audio} soundscapes had no audio (expected at commit time)')
if error_count:
    print(f'⚠️  {error_count} prediction failure(s)')
print(f'\n✅ Generated {len(all_row_ids)} rows')

In [ ]:
# === CELL 8: BUILD & SAVE SUBMISSION ===
probs_matrix = np.concatenate(all_probs_list, axis=0)

_mean_p = probs_matrix.mean()
_std_p  = probs_matrix.std()
if abs(_mean_p - 0.5) < 0.001 and _std_p < 0.01:
    print(f'\n⚠️  WARNING: all predictions near 0.5 (mean={_mean_p:.4f}, std={_std_p:.4f})')
    print('   Check ONNX loading (Cell 4) and checkpoint loading (Cell 3).')
else:
    print(f'✅ Distribution healthy: mean={_mean_p:.4f}, std={_std_p:.4f}')

sub_df = pd.DataFrame(probs_matrix, columns=species)
sub_df.insert(0, 'row_id', all_row_ids)
sample_cols = pd.read_csv(SAMPLE_SUB, nrows=0).columns.tolist()
sub_df = sub_df[sample_cols]

out_path = '/kaggle/working/submission.csv'
sub_df.to_csv(out_path, index=False)
print(f'✅ Submission saved: {out_path}  shape={sub_df.shape}')
print(sub_df.head(3))